In [1]:
import os
import shutil
import random
from pathlib import Path

# ==========================================
# CONFIGURATION
# ==========================================

# --- WINDOWS PATH FIX ---
# Remember to change 'D:\' to whatever drive letter your USB/Drive is assigned
BASE_DIR = Path(r"C:\Users\USER\Documents\GitHub\DeeperHealthBarsLeague\LolHealth-Bar")
SOURCE_DIR = BASE_DIR / "synthetic_dataset"
SRC_IMAGES = SOURCE_DIR / "images"
SRC_LABELS = SOURCE_DIR / "labels"

# New YOLO training directory
OUTPUT_DIR = BASE_DIR / "yolo_training_data"

# Split Ratios (Train, Val, Test) must sum to 1.0
TRAIN_RATIO = 0.80
VAL_RATIO = 0.10
TEST_RATIO = 0.10

YOLO_CLASSES = {
    0: "Player Champion",      
    1: "Ally Champion",        
    2: "Enemy Champion",       
    3: "Ally Turret",          
    4: "Enemy Turret",         
    5: "Epic Monster",         
    6: "Jungle Monster"        
}

# ==========================================
# EXECUTION
# ==========================================
def prepare_dataset():
    print("========================================")
    print(" YOLO DATASET PREPARATION PIPELINE")
    print("========================================\n")

    # 1. Count and validate files
    print("Scanning source directories...")
    if not SRC_IMAGES.exists() or not SRC_LABELS.exists():
        print(f"❌ Error: Source images or labels directory not found at {SOURCE_DIR}")
        return

    # Find all .jpg files
    image_files = [f for f in os.listdir(SRC_IMAGES) if f.endswith('.jpg')]
    
    valid_pairs = []
    missing_labels = 0
    
    for img_file in image_files:
        base_name = os.path.splitext(img_file)[0]
        lbl_file = base_name + ".txt"
        
        # Check if the matching label file exists
        if (SRC_LABELS / lbl_file).exists():
            valid_pairs.append(base_name)
        else:
            missing_labels += 1

    total_valid = len(valid_pairs)
    print(f"✓ Found {len(image_files)} images.")
    print(f"✓ Found {total_valid} complete image-label pairs.")
    if missing_labels > 0:
        print(f"⚠ Warning: {missing_labels} images were missing labels and will be ignored.")

    if total_valid == 0:
        print("❌ Error: No valid data to split. Exiting.")
        return

    # 2. Shuffle and calculate splits
    print("\nShuffling data and calculating splits...")
    random.shuffle(valid_pairs)

    train_count = int(total_valid * TRAIN_RATIO)
    val_count = int(total_valid * VAL_RATIO)
    
    train_data = valid_pairs[:train_count]
    val_data = valid_pairs[train_count:train_count + val_count]
    test_data = valid_pairs[train_count + val_count:]  # The rest goes to test

    print(f"  - Training Set:   {len(train_data)} images ({TRAIN_RATIO*100:.0f}%)")
    print(f"  - Validation Set: {len(val_data)} images ({VAL_RATIO*100:.0f}%)")
    print(f"  - Testing Set:    {len(test_data)} images ({TEST_RATIO*100:.0f}%)")

    # 3. Create YOLO directory structure
    print("\nCreating YOLO directory structure...")
    if OUTPUT_DIR.exists():
        print("⚠ Output directory already exists. Clearing old data...")
        try:
            shutil.rmtree(OUTPUT_DIR)
        except Exception as e:
            print(f"❌ Error clearing directory: {e}. Please delete {OUTPUT_DIR} manually.")
            return
            
    for split in ['train', 'val', 'test']:
        (OUTPUT_DIR / 'images' / split).mkdir(parents=True, exist_ok=True)
        (OUTPUT_DIR / 'labels' / split).mkdir(parents=True, exist_ok=True)

    # 4. Copy files to the new structure
    print("Copying files to splits (this may take a moment)...")
    
    def copy_files(file_list, split_name):
        for base_name in file_list:
            img_src = SRC_IMAGES / f"{base_name}.jpg"
            lbl_src = SRC_LABELS / f"{base_name}.txt"
            
            img_dst = OUTPUT_DIR / 'images' / split_name / f"{base_name}.jpg"
            lbl_dst = OUTPUT_DIR / 'labels' / split_name / f"{base_name}.txt"
            
            shutil.copy(img_src, img_dst)
            shutil.copy(lbl_src, lbl_dst)

    copy_files(train_data, 'train')
    copy_files(val_data, 'val')
    copy_files(test_data, 'test')
    
    print("✓ All files copied successfully.")

    # 5. Generate data.yaml for YOLOv8
    print("\nGenerating data.yaml...")
    yaml_path = OUTPUT_DIR / "data.yaml"
    
    # --- WINDOWS YAML FIX ---
    # .as_posix() safely converts backslashes (\) to forward slashes (/) for the YOLO parser
    safe_output_path = OUTPUT_DIR.absolute().as_posix()
    
    yaml_content = f"""# YOLOv8 Dataset Configuration
path: {safe_output_path}  # dataset root dir
train: images/train  # train images (relative to 'path')
val: images/val      # val images (relative to 'path')
test: images/test    # test images (optional)

# Classes
names:
"""
    for cls_id in sorted(YOLO_CLASSES.keys()):
        yaml_content += f"  {cls_id}: {YOLO_CLASSES[cls_id]}\n"
        
    with open(yaml_path, "w") as f:
        f.write(yaml_content)

    print(f"✓ data.yaml created at {yaml_path}")

    print("\n========================================")
    print(" SUCCESS! Your dataset is ready.")
    print(" You can now start training with:")
    print(f" yolo train data='{yaml_path.absolute().as_posix()}' model=yolov8n.pt epochs=100 imgsz=640")
    print("========================================")

if __name__ == "__main__":
    prepare_dataset()

 YOLO DATASET PREPARATION PIPELINE

Scanning source directories...
✓ Found 50000 images.
✓ Found 50000 complete image-label pairs.

Shuffling data and calculating splits...
  - Training Set:   40000 images (80%)
  - Validation Set: 5000 images (10%)
  - Testing Set:    5000 images (10%)

Creating YOLO directory structure...
Copying files to splits (this may take a moment)...
✓ All files copied successfully.

Generating data.yaml...
✓ data.yaml created at C:\Users\USER\Documents\GitHub\DeeperHealthBarsLeague\LolHealth-Bar\yolo_training_data\data.yaml

 SUCCESS! Your dataset is ready.
 You can now start training with:
 yolo train data='C:/Users/USER/Documents/GitHub/DeeperHealthBarsLeague/LolHealth-Bar/yolo_training_data/data.yaml' model=yolov8n.pt epochs=100 imgsz=640
